# Using a different drawing api 👨‍🎨

The following example shows how `ipysketch_lite` can be used with the [js-draw](https://github.com/personalizedrefrigerator/js-draw) api to access sketches from our notebook.
JS-draw is an existing freehand drawing library for JavaScript and TypeScript.

By extending the [BaseSketchWidget](https://matthewandretaylor.github.io/ipysketch_lite/ipysketch_lite.html#ipysketch_lite.BaseSketchWidget) class images can be stored and updated whenever a change is made in the frontend.

`BaseSketchWidget` is abstract: subclasses must implement `get_template()`.
It provides image synchronization and saving; this custom frontend exports PNG data.

In [ ]:
%pip install ipysketch_lite

In [ ]:
custom_template = """
export default {
  async render({ model, el }) {
    await import("https://cdn.jsdelivr.net/npm/js-draw@1.20.3/dist/bundle.min.js");
    const sketch = document.createElement("div");
    sketch.innerHTML = `<div id="editor"></div>`;
    sketch.style.display = "flex";
    sketch.style.justifyContent = "center";
    el.appendChild(sketch);
    var canvas = document.createElement("canvas");
    var ctx = canvas.getContext("2d");
    const editorElement = sketch.querySelector("#editor");
    editorElement.style.width = "800px";
    editorElement.style.height = "600px";
    const editor = new jsdraw.Editor(editorElement);
    editor.dispatch(editor.image.setAutoresizeEnabled(true), false);
    editor.addToolbar();

    function canvasUpload() {
      model.set('_sketch_data', canvas.toDataURL());
      model.save_changes();
    }

    function madeEdit() {
      var viewport = editor.image.getImportExportViewport();
      canvas.width = viewport.getScreenRectSize().x;
      canvas.height = viewport.getScreenRectSize().y;
      var renderer = new jsdraw.CanvasRenderer(ctx, viewport);
      editor.image.render(renderer, viewport);
      canvasUpload();
    }

    editorElement.addEventListener("mouseup", madeEdit);
    editorElement.addEventListener("touchend", madeEdit);
  }
}
"""

from ipysketch_lite import BaseSketchWidget


class CustomSketch(BaseSketchWidget):
    """
    This includes a template that allows for using the jsdraw library to draw on the sketchpad
    """

    def get_template(self) -> str:
        return custom_template


custom_sketch = CustomSketch()

Draw in the custom editor and release the pointer before running the next
cell. Rerun it after edits to retrieve the current image.

In [ ]:
custom_sketch.image